In [36]:
from rocketcea.cea_obj_w_units import CEA_Obj
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, fields, is_dataclass, MISSING
from typing import dataclass_transform
from pyfluids import Fluid, FluidsList, Input

P_A_BAR = 1.01325  # ambient pressure in bar
g = 9.80665  # standard gravity in m/s^2


def q(unit="", desc="", fmt=".4g", *, default=MISSING, default_factory=MISSING, init=True):
    kw = {}
    if default is not MISSING:
        kw["default"] = default
    if default_factory is not MISSING:
        kw["default_factory"] = default_factory
    return field(metadata={"unit": unit, "desc": desc, "fmt": fmt}, init=init, **kw)

def _fmt(v, meta):
    if v is None:
        return "..."
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return format(v, meta.get("fmt", ".4g"))
    return str(v)

def _rows(obj, indent=0):
    out = []
    for f in fields(obj):
        v = getattr(obj, f.name, None)
        name = "  " * indent + f.name + ("" if f.init else "*")
        if is_dataclass(v):
            out.append((name, "", "", type(v).__name__))
            out += _rows(v, indent + 1)
        else:
            m = f.metadata
            out.append((name, _fmt(v, m), m.get("unit", ""), m.get("desc", "")))
    return out

def _show(self):
    rows = _rows(self)
    wn = max(len(r[0]) for r in rows)
    wv = max(len(r[1]) for r in rows)
    body = "\n".join(f"  {n:<{wn}} {v:>{wv}}  {u:<7} {d}" for n, v, u, d in rows)
    return f"{type(self).__name__}\n{body}"

@dataclass_transform(field_specifiers=(field, q))
def component(cls):
    cls = dataclass(cls)
    cls.__str__ = cls.__repr__ = _show
    return cls

@component
class Propellent:
    cea_name: str = q("-", "RocketCEA name") 
    # E.g. LOX / RP-1 / Isopropanol / CH4 / Kerosene / NitrousOxide
    # https://rocketcea.readthedocs.io/en/latest/propellants.html
    coolprop_fluid: FluidsList
    t_tank: float = q("C", "tank temperature")
    p_tank: float = q("bar", "tank pressure")
    fluid: Fluid = field(init=False)

    def __post_init__(self):
        self.fluid = Fluid(self.coolprop_fluid)



@component
class EngineRequirements:
    F:  float = q("N", "thrust")
    p_a_bar: float = q("bar", "ambient pressure")

@component
class EngineChoices:
    OF: float = q("-", "mixture ratio")
    p_c_bar: float = q("bar", "chamber pressure")
    p_e_bar: float = q("bar", "exit pressure")
    perc_film: float = q("-", "percentage of film cooling (of core total core mdot)")
    fuel_name: str = q("-", "fuel name", default="RP-1")
    oxidizer_name: str = q("-", "oxidizer name", default="LOX")

@component
class InjectorChoices:
    inj_dp: float = q("bar", "injector pressure drop")
    fuel_cd: float = q("-", "fuel injector discharge coefficient")
    ox_cd: float = q("-", "oxidizer injector discharge coefficient")

@component
class ChamberGeometry:
    eps: float = q("-", "area ratio")
    d_t: float = q("m", "throat diameter")
    d_e: float = q("m", "exit diameter")

@component
class InjectorGeometry:
    A_fuelinj: float = q("m^2", "fuel injector area")
    A_oxinj: float = q("m^2", "oxidizer injector area")
    fuel_cd: float = q("-", "fuel injector discharge coefficient")
    ox_cd: float = q("-", "oxidizer injector discharge coefficient")


@component
class EngineCalibration:
    eff_cstar: float = q("-", "c* efficiency", default=0.95)

@component
class EnginePerformance:
    fuel: Propellent
    oxidizer: Propellent
    p_c_bar: float = q("bar", "chamber pressure")
    p_a_bar: float = q("bar", "ambient pressure")
    OF: float = q("-", "mixture ratio")
    F_amb: float = q("N", "delivered thrust")
    F_vac: float = q("N", "delivered thrust in vacuum")
    F_sl: float = q("N", "delivered thrust at sea level")
    isp_amb: float = q("s", "specific impulse at ambient pressure")
    isp_vac: float = q("s", "specific impulse")
    isp_sl: float = q("s", "specific impulse at sea level")
    cstar: float = q("m/s", "characteristic velocity")
    T_c: float = q("K", "chamber temperature")
    mdot: float = q("kg/s", "total mass flow")
    mdot_ox: float = q("kg/s", "oxidiser mass flow")
    mdot_fuel: float = q("kg/s", "fuel mass flow")
    Cf_amb: float = q("-", "thrust coefficient at ambient pressure")
    Cf_vac: float = q("-", "thrust coefficient in vacuum")
    Cf_sl: float = q("-", "thrust coefficient at sea level")
    p_fuel_injdp: float = q("bar", "fuel injector pressure drop")
    p_ox_injdp: float = q("bar", "oxidizer injector pressure drop")
    p_fuel_preinj: float = q("bar", "fuel pre-injector pressure")
    p_ox_preinj: float = q("bar", "oxidizer pre-injector pressure")
    stiffness_fuel: float = q("-", "fuel injector stiffness")
    stiffness_ox: float = q("-", "oxidizer injector stiffness")

@component
class Engine:
    chamber: ChamberGeometry
    injector: InjectorGeometry
    propcea: PropCEA
    calib: EngineCalibration

    def point_performance(self, p_c_bar: float,  p_a_bar: float, OF: float) -> EnginePerformance:
        isp_vac, cstar_theory, t_c = self.propcea.cea.get_IvacCstrTc(Pc=p_c_bar, MR=OF, eps=self.chamber.eps, frozen=1, frozenAtThroat=1)
        isp_sl = self.propcea.cea.estimate_Ambient_Isp(Pc=p_c_bar, MR=OF, eps=self.chamber.eps, Pamb=p_a_bar)[0] * self.calib.eff_cstar
        isp_amb = self.propcea.cea.estimate_Ambient_Isp(Pc=p_c_bar, MR=OF, eps=self.chamber.eps, Pamb=p_a_bar)[0] * self.calib.eff_cstar

        cstar = cstar_theory * self.calib.eff_cstar
        mdot = self.chamber.d_t**2 * np.pi / 4 * p_c_bar * 1e5 / cstar

        mdot_ox = mdot * OF / (1 + OF)
        mdot_fuel = mdot / (1 + OF)

        F_amb = mdot * isp_amb * g
        F_vac = mdot * isp_vac * g
        F_sl = mdot * isp_sl * g

        Cf_amb = F_amb / (p_c_bar * 1e5 * self.chamber.d_t**2 * np.pi / 4)
        Cf_vac = F_vac / (p_c_bar * 1e5 * self.chamber.d_t**2 * np.pi / 4)
        Cf_sl = F_sl / (p_c_bar * 1e5 * self.chamber.d_t**2 * np.pi / 4)

        # Guess parameters for first loop, if they aren't good enough, add a loop to iterate to a better solution

        fuel_preinj = propcea.fuel.fluid.with_state(Input.pressure(p_c_bar * 1e5), Input.temperature(propcea.fuel.t_tank))
        ox_preinj = propcea.oxidizer.fluid.with_state(Input.pressure(p_c_bar * 1e5), Input.temperature(propcea.oxidizer.t_tank))
        
        p_fuel_injdp = (mdot_fuel / self.injector.A_fuelinj / self.injector.fuel_cd)**2 / (2 * fuel_preinj.density) / 1e5
        p_ox_injdp = (mdot_ox / self.injector.A_oxinj / self.injector.ox_cd)**2 / (2 * ox_preinj.density) / 1e5

        p_fuel_preinj = p_c_bar + p_fuel_injdp
        p_ox_preinj = p_c_bar + p_ox_injdp

        stiffness_fuel = p_fuel_injdp / p_c_bar
        stiffness_ox = p_ox_injdp / p_c_bar

        

        return EnginePerformance(
            fuel=propcea.fuel,
            oxidizer=propcea.oxidizer,
            p_c_bar=p_c_bar,
            p_a_bar=p_a_bar,
            OF=OF,
            F_amb=F_amb,
            F_vac=F_vac,
            F_sl=F_sl,
            isp_amb=isp_amb,
            isp_vac=isp_vac,
            isp_sl=isp_sl,
            cstar=cstar,
            T_c=t_c,
            mdot=mdot,
            mdot_ox=mdot_ox,
            mdot_fuel=mdot_fuel,
            Cf_amb=Cf_amb,
            Cf_vac=Cf_vac,
            Cf_sl=Cf_sl,
            p_fuel_injdp=p_fuel_injdp,
            p_ox_injdp=p_ox_injdp,
            p_fuel_preinj=p_fuel_preinj,
            p_ox_preinj=p_ox_preinj,
            stiffness_fuel=stiffness_fuel,
            stiffness_ox=stiffness_ox)

@component
class PropCEA:
    oxidizer: Propellent
    fuel: Propellent

    def __post_init__(self):
        self.cea = CEA_Obj(
            oxName = self.oxidizer.cea_name,
            fuelName = self.fuel.cea_name,
            isp_units='sec',
            cstar_units = 'm/s',
            pressure_units='Bar',
            temperature_units='K',
            sonic_velocity_units='m/s',
            enthalpy_units='J/g',
            density_units='kg/m^3',
            specific_heat_units='J/kg-K',
            viscosity_units='centipoise', # stored value in pa-s
            thermal_cond_units='W/cm-degC', # stored value in W/m-K
            # fac_CR=self.cr,
            make_debug_prints=False)
        
def size_engine(engine_req: EngineRequirements, engine_choices: EngineChoices, injector_choices: InjectorChoices, engine_calib: EngineCalibration, propcea: PropCEA) -> Engine:

    # With low pressure engines, I will hereon assume frozen at throat properties. Refer to notes on non equilibrium flows if this assumption is not valid.
    eps = propcea.cea.get_eps_at_PcOvPe(Pc=engine_choices.p_c_bar, MR=engine_choices.OF, PcOvPe=engine_choices.p_c_bar/engine_choices.p_e_bar, frozen=1, frozenAtThroat=1)
    cstar = propcea.cea.get_Cstar(Pc=engine_choices.p_c_bar, MR=engine_choices.OF) * engine_calib.eff_cstar
    isp_ideal = propcea.cea.estimate_Ambient_Isp(Pc=engine_choices.p_c_bar, MR=engine_choices.OF, eps=eps, Pamb=engine_choices.p_e_bar)[0] * engine_calib.eff_cstar

    # mdot for at the design point
    mdot = engine_req.F / (isp_ideal * g)
    A_t = mdot * cstar / (engine_choices.p_c_bar * 1e5)
    d_t = np.sqrt(4 * A_t / np.pi)  
    d_e = d_t * np.sqrt(eps) 

    chamber = ChamberGeometry(eps=eps, d_t=d_t, d_e=d_e)

    mdot_fuel = mdot / (1 + engine_choices.OF)
    mdot_ox = mdot * engine_choices.OF / (1 + engine_choices.OF)

    p_preinj = engine_choices.p_c_bar + injector_choices.inj_dp
    fuel_preinj = propcea.fuel.fluid.with_state(Input.pressure(p_preinj * 1e5), Input.temperature(propcea.fuel.t_tank))
    ox_preinj = propcea.oxidizer.fluid.with_state(Input.pressure(p_preinj * 1e5), Input.temperature(propcea.oxidizer.t_tank))

    A_fuelinj = mdot_fuel / (injector_choices.fuel_cd * np.sqrt(2 * 1e5 * injector_choices.inj_dp * fuel_preinj.density))
    A_oxinj = mdot_ox / (injector_choices.ox_cd * np.sqrt(2 * 1e5 * injector_choices.inj_dp * ox_preinj.density))

    injector = InjectorGeometry(A_fuelinj=A_fuelinj, A_oxinj=A_oxinj, fuel_cd=injector_choices.fuel_cd, ox_cd=injector_choices.ox_cd)

    return Engine(chamber, injector, propcea, engine_calib)

@component
class PumpRequirements:
    propellant: Propellent
    Q_req: float = q("m^3/s", "required pump flow rate")
    H_req: float = q("m", "required pump head")

    @staticmethod
    def from_engine_perf(engine_perf: EnginePerformance, side: str) -> PumpRequirements:
        if side == "ox":
            propellant = engine_perf.oxidizer
            ox_tank = propellant.fluid.with_state(Input.pressure(engine_perf.p_ox_preinj * 1e5), Input.temperature(propellant.t_tank))
            Q_req = engine_perf.mdot_ox / ox_tank.density
            H_req = engine_perf.p_ox_preinj
        elif side == "fuel":
            propellant = engine_perf.fuel
            fuel_tank = propellant.fluid.with_state(Input.pressure(engine_perf.p_fuel_preinj * 1e5), Input.temperature(propellant.t_tank))
            Q_req = engine_perf.mdot_fuel / fuel_tank.density
            H_req = engine_perf.p_fuel_preinj
        else:
            raise ValueError(f"Invalid pump side: {side}")
        
        return PumpRequirements(propellant=propellant, Q_req=Q_req, H_req=H_req)

@component
class PumpChoices:
    RPM: float = q("RPM", "pump rotational speed")
    n_blades: int = q("-", "number of impeller blades")
    v_0: float = q("m/s", "pump inlet velocity", default=1.0)
    d1_over_d0: float = q("-", "impeller inlet diameter over pump inlet diameter", default=1.1)
    flow_coeff_throat: float = q("-", "flow coefficient at pump throat", default=0.8)
    coeff_p: float = q("-", "Barske's phi coefficient, only used for Barske sizing method", default=0.2)

@component
class PumpGeometry:
    d_0: float = q("m", "Pump inlet diameter")
    d_1: float = q("m", "Impeller inlet diameter")
    d_2: float = q("m", "Impeller outlet diameter")
    d_3: float = q("m", "Casing diameter")
    b_1: float = q("m", "Impeller inlet blade height")
    b_2: float = q("m", "Impeller outlet blade height")
    b_3: float = q("m", "Casing height")
    s_ax: float = q("m", "Axial spacing between impeller and casing")
    d_throat: float = q("m", "Diffuser throat diameter")
    d_outlet: float = q("m", "Pump outlet diameter")


def size_pump(pump_req: PumpRequirements, pump_choices: PumpChoices, sizing_method: str) -> PumpGeometry:
    d_0 = np.sqrt(4 * pump_req.Q_req / (np.pi * pump_choices.v_0))
    d_1 = d_0 * pump_choices.d1_over_d0
    omega = pump_choices.RPM * 2 * np.pi / 60


    if sizing_method not in ["barske", "lock"]:
        raise ValueError(f"Invalid sizing method: {sizing_method}")

    elif sizing_method == "barske":
        d_2 = np.sqrt((1 / (1 + pump_choices.coeff_p)) * (8 * pump_req.H_req * g / omega**2 + d_1**2))
        u_2 = omega * d_2 / 2
        v_3 = pump_choices.flow_coeff_throat * u_2
        d_throat = np.sqrt(4 * pump_req.Q_req / (np.pi * v_3))
        d_outlet = 2 * d_throat # Arbitrary expansion ratio, doesn't matter too much
        s_ax = min(0.00075, d_2 / 100)
        b_1 = d_1 / 4
        b_2 = max(b_1 * d_1 / d_2, s_ax)

    elif sizing_method == "lock":

        # Temporary lock curve implementation, to be moved into private constants later

        import scipy.interpolate as intrp
        import scipy.optimize as opt

        class _lock_curves:
            def __init__(self):
                self.eta_losses = 0.194
                self.K_factor = 0.17
                """Digitized Lock Fig 10a (h_0 slip factor) and 10b (C_h) for radial blades.
                Returns (h_0(r_ratio, n_blades), C_h(blade_spacing/length))."""
                r_ratio = np.array([0, 0.2, 0.4, 0.6, 0.8, 1.0])
                n_blades = np.array([2, 4, 8, 16])
                h0 = np.transpose([
                    [0.498, 0.482, 0.435, 0.334, 0.190, 0.0],   # 2 blades
                    [0.636, 0.636, 0.606, 0.517, 0.332, 0.0],   # 4 blades
                    [0.768, 0.768, 0.768, 0.731, 0.518, 0.0],   # 8 blades
                    [0.864, 0.864, 0.864, 0.864, 0.753, 0.0]])  # 16 blades
                h0_2D = intrp.RegularGridInterpolator((r_ratio, n_blades), h0, method="pchip")
                h0_16 = intrp.PchipInterpolator(r_ratio, [0.864, 0.864, 0.864, 0.864, 0.753, 0.0])
                self.h_0_grid = lambda r, n: float(h0_2D([[r, n]])[0]) if n <= 16 else float(h0_16(r))
                bsl = [0.506, 0.606, 0.704, 0.804, 0.904, 1.003, 1.104, 1.202, 1.301, 1.401,
                        1.501, 1.603, 1.704, 1.803, 1.903, 2.003, 2.104, 2.206, 2.307, 2.406, 2.505]
                Ch = [0.995, 0.994, 0.988, 0.980, 0.967, 0.955, 0.942, 0.926, 0.908, 0.890,
                        0.872, 0.854, 0.837, 0.819, 0.803, 0.787, 0.772, 0.757, 0.743, 0.729, 0.717]
                Ch_i = intrp.interp1d(bsl, Ch, kind="linear", bounds_error=False, fill_value="extrapolate")
                self.C_h_grid = lambda s: float(np.clip(Ch_i(s), 0.0, 1.0))
            def h_0(self, r_ratio: float, n_blades: int) -> float:
                return self.h_0_grid(r_ratio, n_blades)
            def C_h(self, blade_spacing_over_length: float) -> float:
                return self.C_h_grid(blade_spacing_over_length)

        _lock = _lock_curves()
        
        d_2_estimate = (2 / omega * np.sqrt(2 * g * pump_req.H_req  / 1.4)) # Guess of a head coefficient around 1.4
        def size_from_d2(d_2):
            u_2 = omega * d_2 / 2
            v_3 = pump_choices.flow_coeff_throat * u_2
            d_throat = np.sqrt(4 * pump_req.Q_req / (np.pi * v_3))
            d_outlet = 2 * d_throat # Arbitrary expansion ratio, doesn't matter too much

            h_0 = _lock.h_0(d_1 / d_2, pump_choices.n_blades)
            C_h = _lock.C_h((d_1 * np.pi / pump_choices.n_blades) / (d_2 - d_1 * 1.1)/2) # 1.1 to not divide by zero

            dummy_1a = h_0 * u_2**2 / g
            dummy_1b = _lock.eta_losses * 24 / (g * d_throat**4 * np.pi**2) * (1 - (d_throat / d_outlet)**2)**2
            dummy_1c = 24 * (d_outlet**-4 - d_0**-4) / (np.pi**2 * g)
            Q_ops = np.sqrt(dummy_1a / (dummy_1b + dummy_1c))

            s_ax = min(0.00075, d_2 / 100)
            b_1 = d_1 / 4
            b_2 = max(b_1 * d_1 / d_2, s_ax)
            v_r_ratio = (b_1*d_1) / (b_2*d_2)

            H_total = dummy_1a - C_h * _lock.K_factor * v_r_ratio * (u_2**2 / g) * (d_1 / d_2) * (1 - pump_req.Q_req / Q_ops)
            H_static = H_total - dummy_1c / 3 * pump_req.Q_req**2
            return H_static, d_0, d_1, d_2, b_1, b_2, s_ax, d_throat, d_outlet

        d_2 = opt.toms748(lambda d_2: pump_req.H_req - size_from_d2(d_2)[0], d_1*1.1, d_2_estimate * 2)
        _, d_0, d_1, d_2, b_1, b_2, s_ax, d_throat, d_outlet = size_from_d2(d_2)
    
    b_3 = b_2 + 2 * s_ax
    d_3 = d_2 + max(b_3, 0.0025)
    return PumpGeometry(d_0=d_0, d_1=d_1, d_2=d_2, d_3=d_3, b_1=b_1, b_2=b_2, b_3=b_3, s_ax=s_ax, d_throat=d_throat, d_outlet=d_outlet)




    


lox = Propellent("LOX", FluidsList.Oxygen, t_tank=-183, p_tank=3)
rp1 = Propellent("RP-1", FluidsList.nDodecane, t_tank=20, p_tank=3)

propcea = PropCEA(lox, rp1)
engine_req = EngineRequirements(F=12000, p_a_bar=1.01325)
engine_choices = EngineChoices(OF=2, p_c_bar=50, p_e_bar=1, perc_film=0.1)
injector_choices = InjectorChoices(inj_dp=10, fuel_cd=0.8, ox_cd=0.8)
engine_calib = EngineCalibration(eff_cstar=0.95)
engine = size_engine(engine_req, engine_choices, injector_choices, engine_calib, propcea)
engine_perf = engine.point_performance(p_c_bar=50, p_a_bar=1.01325, OF=2)
# print(engine_perf)
# req.check(engine_perf)
ox_pump_req = PumpRequirements.from_engine_perf(engine_perf, "ox")
ox_pump_choices = PumpChoices(RPM=20000, n_blades=6)
ox_pump_geometry = size_pump(ox_pump_req, ox_pump_choices, sizing_method="lock")
# pchoices = PumpChoices(RPM=20000)
# pcalib = PumpCalibration()
# mech = Mechanicals()
# pgeom = size_pump(ox_req, pchoices, pcalib)
# pump = Pump(pgeom, lox, mech, pcalib)
# pres = pump.analyse(ox_req.Q, pchoices.RPM)
# ox_req.check(pres)

# print(engine)

ValueError: a and b are not an interval [0.06999389425649935, 0.05538001430710092]